# Capstone: integrated MedLIME-inspired explanations

**MedLIME Learning Track — Notebook 12/12** · Curriculum weeks 33–40 · Suggested time: 60–100 minutes

**Before starting:** All previous notebooks.

**Learning goal:** Compare controlled explanation variants and prepare a critical replication plan.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## The four-step research workflow
1. Create a **reference-only** library of positive synthetic cases.
2. Train a lightweight image inpainter on separate synthetic images.
3. For each held-out query, compare black LIME, generative LIME, regularization, adaptation, and combinations.
4. Evaluate AUPRC against known toy lesion masks and inspect failures.

This is an **educational integration exercise, not a numerical reproduction** of the MedLIME CVPR 2026 tables. It substitutes synthetic images, a hand-specified predictor, a small CNN inpainter (not the paper MAE), simplified supervised single-image affine adaptation (using true test label), and a simplified evidence penalty. The *FULL* variant has optimistic test-time label access.

In [ ]:
# Partition data by separate seed ranges. No query appears in reference or training pool.
reference=np.stack([synthetic_image(5000+i,True)[0] for i in range(16)])
reference_coef=np.asarray([lime_explain(z,n=75,seed=11)['coef'] for z in reference])
queries=[synthetic_image(7000+i,True) for i in range(6)]
inpainter,losses=train_inpainter(n=96,epochs=8,seed=10)
print('Queries:',len(queries),'reference images:',len(reference),'final inpainter loss:',round(losses[-1],4))

## Run fair paired ablations
Within each query use the *same random mask seed and number of masks* across arms. The local optimization is small; to keep runtime low, we use 100 perturbations and only six held-out toy lesions. These figures are too small for a clinical claim.

In [ ]:
arms=['LIME (black)','GM only','GM + EBR','GM + supervised adaptation','ALL (GM+STTA+EBR)']
records=[]
for i,(original,lesion) in enumerate(queries):
    q=original*.82
    prior,_,_=evidence_prior(q,reference,reference_coef,k=4)
    adapter,_=supervised_adapt(q,true_label=1,steps=28)
    configs=[dict(fill='zero'),
             dict(fill='learned',inpainter=inpainter),
             dict(fill='learned',inpainter=inpainter,prior=prior,l1=.002,l2=.1),
             dict(fill='learned',inpainter=inpainter,adapt=adapter),
             dict(fill='learned',inpainter=inpainter,prior=prior,l1=.002,l2=.1,adapt=adapter)]
    scores=[]
    for config in configs:
        explanation=lime_explain(q,n=100,seed=26,**config)
        scores.append(auprc(lesion,explanation['map']))
    records.append(scores)
    print('Query',i+1,dict(zip(arms,np.round(scores,3))))
results=np.asarray(records)
assert results.shape == (6,5)
print('\nMean AUPRC (illustrative only):')
for arm,v in zip(arms,results.mean(axis=0)): print(f'{arm:>28}  {v:.3f}')

## Read an ablation table as a scientist
Ablation asks what changes when one component is removed, **holding other relevant choices fixed**. On this tiny dataset, variation across individual examples is often more informative than a mean. Do not claim the last column is best before inspecting it.

In [ ]:
fig,ax=plt.subplots(figsize=(8,3.5))
ax.boxplot([results[:,j] for j in range(len(arms))])
ax.set_xticks(range(1,6),labels=['LIME','GM','GM+EBR','GM+STTA','ALL'])
ax.set_ylabel('Pixel AUPRC');ax.set_title('Synthetic cases — not paper results');ax.set_ylim(0,1)
plt.xticks(rotation=15);plt.tight_layout();plt.show()
print('Top arm by observed mean:',arms[int(np.argmax(results.mean(0)))])
print('Mean is not an unbiased test of tuned methods at n=6.')

## Build your real-paper reproduction plan
Before using real patient images, students should read the paper and supplementary protocol, check dataset licensing and ethical permissions, isolate patients across splits, identify exactly what supervision STTA requires, identify the pretraining corpus/architecture for the actual MAE, implement paper-specific neighbor similarity and EBR terms, match mask counts and hyperparameters, and compare to every relevant baseline on the same test data.

**Read:** [MedLIME CVPR 2026](https://openaccess.thecvf.com/content/CVPR2026/html/Magazine_MedLIME_A_Distribution-Aligned_and_Evidence-Supported_Framework_for_Medical_Saliency_Explanations_CVPR_2026_paper.html) and its supplement. **Do not use label-informed adaptation results as label-blind evaluation**.

In [ ]:
# Challenge template: document results without inventing successful outcomes.
for arm,mean,std in zip(arms,results.mean(0),results.std(0,ddof=1)):
 print(f'{arm:>28}: mean={mean:.3f}, SD={std:.3f}, n={len(queries)}')
print('Research interpretation: explain failures, do not only rank averages.')

## What you should now understand
- The integrated pipeline combines perturbation, reconstruction, adaptation, and evidence priors.
- Proper ablations must control masks, queries, supervision, and evaluation metric.
- Synthetic demonstrations cannot verify the paper's clinical performance claims.

## Try it yourself (do not just read the code)
1. Create a component-ablation matrix and defend every experimental control.
2. Run three different sets of six held-out queries and report uncertainty.
3. Write a two-page methods audit of the real MedLIME paper, identifying any toy/paper mismatch and how you would close it.

## Exit ticket
**Final defense:** Explain in your own words why each component exists, which assumptions it makes, and what data could falsify its promised benefit.

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.